<a href="https://colab.research.google.com/github/farrelrassya/IntroductionMachineLearningwithpython/blob/main/01.Common_Conventions_and_API_Elements_of_scikit_learn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bab 1 - Konvensi Umum dan Elemen API scikit-learn (Common Conventions and API Elements of scikit-learn)

Notebook ini berisi penjelasan komprehensif mengenai **Bab 1 Buku "scikit-learn Cookbook, Third Edition"** karya John Sukup. Materi ini mencakup filosofi desain scikit-learn, konsep OOP (Object-Oriented Programming) yang digunakan, serta elemen-elemen API utama seperti Estimator, Transformer, Pipeline, atribut model, dan metadata routing. Seluruh penjelasan disajikan dalam **Bahasa Indonesia** dengan standar premium untuk memudahkan proses pembelajaran.

## 1. Filosofi Desain scikit-learn

Desain pustaka `scikit-learn` didasarkan pada beberapa prinsip utama:
* **Konsistensi (Consistency):** Semua objek (baik model prediktif maupun pemroses data) berbagi antarmuka yang seragam. Metode utama seperti `fit()`, `predict()`, dan `transform()` digunakan secara konsisten di seluruh pustaka.
* **Kesederhanaan (Simplicity):** Parameter model mudah diatur langsung saat inisialisasi model, dengan nilai bawaan (default) yang masuk akal.
* **Modularitas (Modularity):** Komponen-komponen machine learning (seperti preprocessing, reduksi dimensi, dan estimasi) dapat digabungkan dengan mudah seperti balok mainan.
* **Dapat Digunakan Kembali (Reusability):** Algoritma beroperasi langsung pada struktur data standar Python, NumPy array, SciPy sparse matrices, atau Pandas DataFrame, tanpa memerlukan format data kustom.

## 2. Memahami Estimator

Dalam scikit-learn, **Estimator** adalah objek apa pun yang dapat mempelajari sesuatu dari data. Proses pembelajaran ini dilakukan melalui metode `fit()`. Estimator dapat berupa model prediksi (klasifikasi, regresi) maupun transformer (pembersihan/skalasi data).

Dua metode utama pada Estimator model adalah:
1. `fit(X, y)`: Melatih model pada data fitur $X$ dan target $y$ (untuk supervised learning).
2. `predict(X)`: Memprediksi nilai target untuk data baru berdasarkan model yang telah dilatih.

Mari kita lihat contoh implementasi sederhana menggunakan model regresi linier (`LinearRegression`):

In [1]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Membuat dataset contoh sederhana
X = np.array([[1], [2], [3], [4], [5]])  # Fitur (matriks 2D)
y = np.array([1.2, 1.8, 3.1, 3.9, 5.0])  # Target (array 1D)

# Inisialisasi estimator model
model = LinearRegression()

# Melatih model menggunakan metode fit()
model.fit(X, y)

# Memprediksi data baru menggunakan metode predict()
X_new = np.array([[6], [7]])
predictions = model.predict(X_new)

print("Data baru:", X_new.flatten())
print("Hasil Prediksi:", predictions)

Data baru: [6 7]
Hasil Prediksi: [5.91 6.88]


### Metode Pintasan: `fit_predict()`

Untuk algoritma pembelajaran tanpa pengawasan (*unsupervised learning*) seperti pengelompokan (*clustering*), kita sering kali ingin langsung mencocokkan model dan memprediksi cluster data pada saat yang sama. Di sinilah metode `fit_predict()` digunakan:

In [2]:
from sklearn.cluster import KMeans

# Dataset contoh untuk clustering
X_cluster = np.array([[1], [1.5], [3], [10], [11]])

# Inisialisasi model KMeans dengan 2 kluster
kmeans = KMeans(n_clusters=2, random_state=42)

# Melatih model dan memprediksi label kluster sekaligus
labels = kmeans.fit_predict(X_cluster)

print("Data:", X_cluster.flatten())
print("Label Kluster:", labels)

Data: [ 1.   1.5  3.  10.  11. ]
Label Kluster: [0 0 0 1 1]


## 3. Transformer dan Metode `transform()`

**Transformer** adalah estimator khusus yang memodifikasi atau menyaring data (biasanya digunakan untuk preprocessing data). Objek ini melatih parameter transformasi menggunakan `fit()`, lalu mengubah data menggunakan metode `transform()`.

Metode utama pada Transformer adalah:
* `fit(X)`: Mempelajari parameter transformasi (misal: nilai rata-rata dan standar deviasi pada penyekalaan data).
* `transform(X)`: Menerapkan transformasi ke data $X$.
* `fit_transform(X)`: Metode pintasan untuk menjalankan `fit` dan `transform` sekaligus pada data pelatihan.

> [!WARNING]
> **Aturan Penting Data Leakage (Kebocoran Data):**
> Selalu jalankan `fit_transform()` hanya pada **data pelatihan (training data)**. Untuk **data pengujian (test data)**, Anda hanya boleh memanggil `transform()`. Memanggil `fit()` pada data pengujian akan menyebabkan parameter pengujian bocor ke model, menghasilkan evaluasi akurasi yang terlalu optimis dan tidak realistis.

In [3]:
from sklearn.preprocessing import StandardScaler

# Data pelatihan dan pengujian
X_train = np.array([[1.0, 2.0], [2.0, 4.0], [3.0, 6.0]])
X_test = np.array([[2.5, 5.0]])

# Inisialisasi StandardScaler
scaler = StandardScaler()

# Cocokkan parameter (fit) dan transformasikan data pelatihan sekaligus
X_train_scaled = scaler.fit_transform(X_train)

# Transformasikan data pengujian menggunakan parameter yang telah dipelajari dari data pelatihan
X_test_scaled = scaler.transform(X_test)

print("Data pelatihan sebelum skalasi:\n", X_train)
print("Data pelatihan setelah skalasi:\n", X_train_scaled)
print("\nData pengujian setelah skalasi (menggunakan parameter train):\n", X_test_scaled)

Data pelatihan sebelum skalasi:
 [[1. 2.]
 [2. 4.]
 [3. 6.]]
Data pelatihan setelah skalasi:
 [[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]

Data pengujian setelah skalasi (menggunakan parameter train):
 [[0.61237244 0.61237244]]


## 4. Membuat Estimator dan Transformer Kustom

Salah satu kelebihan scikit-learn adalah kemudahan pembuatan pemroses data kustom yang dapat langsung dipasang ke dalam Pipeline. Kita dapat mewarisi (subclassing) dua kelas dasar:
1. `BaseEstimator`: Memberikan akses gratis ke metode pengaturan parameter seperti `get_params()` dan `set_params()` tanpa harus menulis `*args` atau `**kwargs` di konstruktor.
2. `TransformerMixin`: Memberikan implementasi metode `fit_transform()` secara otomatis jika kita telah menentukan metode `fit()` dan `transform()`.

Berikut adalah contoh transformer kustom yang berfungsi menghapus kolom yang memiliki deviasi standar di bawah ambang batas tertentu (penyaringan varians rendah):

In [4]:
from sklearn.base import BaseEstimator, TransformerMixin

class VarianceThresholdFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.1):
        self.threshold = threshold
        self.selected_features_ = None
        
    def fit(self, X, y=None):
        # Menghitung standar deviasi di setiap kolom
        std_devs = np.std(X, axis=0)
        # Menyimpan indeks kolom yang memiliki varians di atas ambang batas
        self.selected_features_ = np.where(std_devs >= self.threshold)[0]
        return self
        
    def transform(self, X):
        # Memeriksa apakah model sudah dilatih (opsional, menggunakan check_is_fitted dari sklearn)
        if self.selected_features_ is None:
            raise ValueError("Transformer belum dilatih (fit)!")
        # Mengembalikan kolom yang terpilih saja
        return X[:, self.selected_features_]

# Uji coba transformer kustom
X_dummy = np.array([
    [1.0, 10.0, 0.05],
    [2.0, 10.1, 0.05],
    [3.0, 9.9,  0.05]
])
# Kolom ketiga memiliki std = 0 (konstan). Kolom pertama std = 0.81, kolom kedua std = 0.08

filter_transformer = VarianceThresholdFilter(threshold=0.1)
X_filtered = filter_transformer.fit_transform(X_dummy)

print("Data Asli (3 kolom):\n", X_dummy)
print("Data Tersaring (hanya kolom pertama karena std >= 0.1):\n", X_filtered)

Data Asli (3 kolom):
 [[ 1.   10.    0.05]
 [ 2.   10.1   0.05]
 [ 3.    9.9   0.05]]
Data Tersaring (hanya kolom pertama karena std >= 0.1):
 [[1.]
 [2.]
 [3.]]


## 5. Pipeline dan Otomatisasi Alur Kerja

Kelas `Pipeline` memungkinkan kita merangkai beberapa langkah preprocessing (transformer) dan langkah pemodelan akhir (estimator/model) ke dalam satu objek tunggal. Hal ini menyederhanakan kode, menjaga kebersihan alur kerja, dan mencegah kebocoran data (*data leakage*) selama proses evaluasi silang (*cross-validation*).

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# Membuat pipeline yang terdiri dari StandardScaler (skalasi) dan LinearRegression (model prediktif)
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

# Data latih
X_train = np.array([[10.0], [20.0], [30.0]])
y_train = np.array([2.5, 4.5, 6.5])

# Melatih seluruh pipeline (StandardScaler akan di-fit_transform, hasil skalasi dioper ke fit pada LinearRegression)
pipeline.fit(X_train, y_train)

# Memprediksi data baru menggunakan pipeline
X_new = np.array([[40.0]])
pred = pipeline.predict(X_new)
print(f"Prediksi untuk input {X_new[0][0]}: {pred[0]:.2f}")

Prediksi untuk input 40.0: 8.50


## 6. Atribut Umum dan Metode Evaluasi Model

Setelah sebuah model estimator dilatih (fit), scikit-learn menyimpan parameter yang telah dipelajari sebagai **Atribut Terlatih (Estimated Attributes)**. Atribut ini selalu ditandai dengan garis bawah di akhir namanya (`_`), seperti `coef_` (koefisien bobot) dan `intercept_` (bias/intersep).

Selain itu, estimator model prediktif memiliki metode `score(X, y)` untuk mengevaluasi performa model dengan metrik evaluasi bawaan (R-squared untuk regresi, akurasi untuk klasifikasi).

In [6]:
# Mengakses koefisien dan intersep regresi linier dari model di dalam pipeline
reg_model = pipeline.named_steps['regressor']
print("Koefisien (Bobot):", reg_model.coef_)
print("Intersep (Bias):", reg_model.intercept_)

# Evaluasi model menggunakan R-squared (score)
r2_score = pipeline.score(X_train, y_train)
print("Model R2 Score (Evaluasi Latih):", r2_score)

Koefisien (Bobot): [1.63299316]
Intersep (Bias): 4.5
Model R2 Score (Evaluasi Latih): 1.0


## 7. Metadata dan Metadata Routing (Fitur scikit-learn Modern)

Pada versi scikit-learn terbaru (v1.5+), diperkenalkan fitur **Metadata Routing**. Fitur ini memungkinkan pengguna meneruskan informasi tambahan di luar matriks fitur $X$ (seperti bobot sampel `sample_weight` atau label grup `groups`) secara aman dan terotomatisasi melalui seluruh langkah pipeline ke komponen yang membutuhkannya.

Secara internal, tag model (`_get_tags()`) digunakan oleh scikit-learn untuk mengenali kemampuan model (misalnya, apakah model tersebut dapat menangani nilai kosong atau klasifikasi multi-output) agar eksekusi pipeline dapat dioptimalkan secara dinamis.

---

## Ringkasan dan Pandangan (Summary and Outlook)

### 1. Ringkasan Elemen API
Dalam bab ini, kita telah mempelajari dasar-dasar API scikit-learn:
* **Estimator** menggunakan metode `fit()` untuk mempelajari pola dari data.
* **Transformer** menggunakan `fit()`, `transform()`, dan `fit_transform()` untuk melakukan prapemrosesan data secara konsisten tanpa kebocoran data (*data leakage*).
* **Custom Transformer** dapat dengan mudah dibuat menggunakan kelas warisan `BaseEstimator` dan `TransformerMixin` agar kompatibel dengan seluruh pustaka scikit-learn.
* **Pipeline** merangkai preprocessing dan pemodelan dalam satu kesatuan alur kerja yang rapi.
* **Estimated Attributes** yang memiliki akhiran tanda garis bawah (`_`) menyimpan parameter hasil latihan yang siap diinterpretasikan.

### 2. Pandangan ke Depan
Dengan memahami konvensi API scikit-learn di Bab 1 ini, kita sekarang siap masuk ke **Bab 2: Pre-Model Workflow and Data Preprocessing**. Di bab selanjutnya, kita akan mulai menerapkan transformer bawaan scikit-learn secara praktis untuk membersihkan, menyekalakan, dan memodifikasi data riil agar siap dikonsumsi oleh model-model pembelajaran mesin.